# Análisis Territorial - Machine Learning
Este cuaderno demuestra el proceso de entrenamiento de modelos de Machine Learning para el análisis de territorios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from src.data_prep import load_and_clean_data

# Cargar datos
df = load_and_clean_data('../')
df.head()

## Visualización de Clusters
Cargamos el modelo entrenado y visualizamos cómo se agrupan las zonas.

In [ ]:
models_dir = 'models'
kmeans = joblib.load(os.path.join(models_dir, 'clustering_model.joblib'))
scaler = joblib.load(os.path.join(models_dir, 'scaler.joblib'))

features = [
    'tasa_alfabetismo', 'cobertura_primaria', 'cobertura_secundaria',
    'poblacion_total', 'indice_pobreza', 'tasa_desempleo',
    'acceso_agua_potable', 'acceso_electricidad', 'acceso_internet'
]

X_scaled = scaler.transform(df[features])
df['cluster'] = kmeans.predict(X_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='indice_pobreza', y='acceso_internet', hue='cluster', palette='viridis', s=100)
plt.title('Clusters de Zonas: Pobreza vs Internet')
plt.show()

## Predicción de Score Territorial
Usamos el modelo de regresión para predecir el score de oportunidad.

In [ ]:
reg = joblib.load(os.path.join(models_dir, 'regression_model.joblib'))
df['predicted_score'] = reg.predict(X_scaled)

df_results = df[['zone_name', 'cluster', 'predicted_score']].sort_values(by='predicted_score', ascending=False)
df_results